# COMPASS diagnostics -- why does `profile_data` yield fewer patients than `ALL_2025_03`?

`profile_data` merges 7 OncDRS releases (`ALL_2021_11` .. `ALL_2026_03`) including
`ALL_2025_03` itself, so at the row level it is a **superset** of `baseline`. It
should yield *more* patients. It yields fewer. This notebook localizes that loss
to a specific stage and a specific predicate.

**Strictly read-only.** Nothing under either data root is written. Any scratch
output goes to `DIAG_OUT` (section 0).

Prime suspect, verified on polars 1.43.2 before writing this notebook:
`pl.Series.str.to_datetime(strict=False)` infers **one** format from the data and
nulls every value not matching it -- it does *not* parse per-row.

```
['2024-01-02','03/04/2024','2024-05-06 07:08:09'] -> [2024-01-02, None, 2024-05-06 07:08:09]
['03/04/2024','2024-01-02']                       -> [2024-04-03, None]   # silent D/M swap
['01-FEB-2024','02-MAR-2024']                     -> ComputeError
['2024-01-02 00:00:00.000','2024-01-02']          -> both parse           # ISO variants safe
```

A single release is format-homogeneous, so one inferred format covers it. A
7-release merge need not be. `coerce_mixed_datetime`'s docstring claims it handles
mixed formats; it does not.

Sections:
0. Setup and provenance
1. Top-level funnel diff (reuses `cp.compare_to_baseline`)
2. Raw source comparison, before any pipeline logic
3. Date-parse audit  <-- prime suspect
4. Stage 1 funnel reconstruction, per predicate
5. Per-MRN attribution: where exactly did each lost patient die?
6. Summary

Each section ends with a **VERDICT** line.

## Section 0 -- Setup and provenance

Establish that both runs exist, are comparable, and neither is stale.

In [ ]:
DATA_VARIANT   = "profile_data"
BASELINE_VARIANT = "baseline"
ARM            = "adt"

import sys, os, json, datetime as _dt
from pathlib import Path

sys.path.insert(0, ".")
import polars as pl
import pandas as pd
import numpy as np

import compass_pipeline as cp

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 250)

RUNS = cp.make_runs(DATA_VARIANT, [ARM])
RUN  = RUNS[0]

# Scratch output -- deliberately NOT under either data root.
DIAG_OUT = Path("/data/gusev/USERS/jpconnor/data/CAIA/diagnostics")
DIAG_OUT.mkdir(parents=True, exist_ok=True)

PROFILE_ROOT  = cp.DATA_VARIANTS[DATA_VARIANT]["data_root"]
BASELINE_ROOT = cp.DATA_VARIANTS[BASELINE_VARIANT]["data_root"]
print(f"\nprofile_data root: {PROFILE_ROOT}")
print(f"baseline     root: {BASELINE_ROOT}")
print(f"diagnostics  out : {DIAG_OUT}")

In [ ]:
def _stat(p):
    """(exists, size_bytes, mtime) for a path, without raising."""
    p = Path(p)
    if not p.exists():
        return False, None, None
    st = p.stat()
    return True, st.st_size, _dt.datetime.fromtimestamp(st.st_mtime)

rows = []
for variant in (BASELINE_VARIANT, DATA_VARIANT):
    for table, path in cp.DATA_VARIANTS[variant]["sources"].items():
        ok, size, mtime = _stat(path)
        rows.append({
            "variant": variant, "table": table, "exists": ok,
            "size_MB": round(size / 1e6, 1) if size else None,
            "mtime": mtime, "path": str(path),
        })
sources_df = pd.DataFrame(rows)
print("=== RAW SOURCES ===")
display(sources_df)

missing = sources_df.loc[~sources_df["exists"]]
if len(missing):
    print("\n!! MISSING RAW SOURCES -- sections below will be incomplete:")
    display(missing[["variant", "table", "path"]])

In [ ]:
# Output artifacts in both roots. Confirms both runs completed and neither is stale.
ARTIFACTS = [
    ("stage1 icd",          "prostate_icd_data.csv"),
    ("stage1 cohort",       f"prostate_{ARM}_survival_cohort_{ARM}.csv"),
    ("stage2 longitudinal", f"longitudinal_prediction_data_{ARM}.csv"),
    ("stage2 attrition",    "cohort_attrition.json"),
    ("stage2 cache",        f"consolidated_longitudinal_data_{ARM}.parquet"),
    ("mrn_lists adt",       f"mrn_lists/{ARM}_mrns.csv"),
    ("mrn_lists platinum",  "mrn_lists/platinum_MRN_list.csv"),
    ("mrn_lists flags",     "mrn_lists/icd_prostate_mrn_flags.csv"),
    ("stage3 attrition",    f"survival_analysis/prediction_inputs_{ARM}/landmark_attrition.json"),
    ("stage3 manifest",     f"survival_analysis/prediction_inputs_{ARM}/build_manifest.json"),
    ("stage3 availability", f"survival_analysis/prediction_inputs_{ARM}/landmark_mrn_availability.csv"),
]

rows = []
for label, rel in ARTIFACTS:
    row = {"artifact": label, "rel_path": rel}
    for name, root in (("baseline", BASELINE_ROOT), ("profile", PROFILE_ROOT)):
        ok, size, mtime = _stat(Path(root) / rel)
        row[f"{name}_exists"] = ok
        row[f"{name}_mtime"] = mtime
    rows.append(row)

artifacts_df = pd.DataFrame(rows)
print("=== OUTPUT ARTIFACTS (both roots) ===")
display(artifacts_df)

_missing_both = artifacts_df.loc[~artifacts_df["baseline_exists"] | ~artifacts_df["profile_exists"]]
if len(_missing_both):
    print("\nNOTE: artifacts absent in one root -- affected sections will degrade gracefully:")
    display(_missing_both[["artifact", "baseline_exists", "profile_exists"]])
else:
    print("\nBoth roots fully populated.")

In [ ]:
# Snapshot mtimes under both data roots so the read-only guarantee can be
# verified at the end of the notebook (verification step 2).
def _mtime_snapshot(root):
    root = Path(root)
    snap = {}
    if not root.exists():
        return snap
    for p in root.rglob("*"):
        if p.is_file():
            try:
                snap[str(p)] = p.stat().st_mtime
            except OSError:
                pass
    return snap

MTIME_SNAPSHOT = {
    "baseline": _mtime_snapshot(BASELINE_ROOT),
    "profile":  _mtime_snapshot(PROFILE_ROOT),
}
print(f"Snapshotted {len(MTIME_SNAPSHOT['baseline'])} baseline + "
      f"{len(MTIME_SNAPSHOT['profile'])} profile_data files for the "
      f"read-only check in section 6.")

## Section 1 -- Top-level funnel diff

Reuses `cp.compare_to_baseline`, which already computes per-stage counts with
deltas. This is the map: the first metric where `pct_change` goes sharply
negative names the stage to attack.

In [ ]:
comparison_df = cp.compare_to_baseline(DATA_VARIANT, RUNS)
if not comparison_df.empty:
    display(comparison_df)

In [ ]:
# Aligned funnel across all three stages, both roots, in one table.
def _load_json(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text())
    except Exception as e:
        print(f"  could not parse {path}: {e}")
        return None

def _stage_counts(root):
    root = Path(root)
    out = {}

    cohort_csv = root / f"prostate_{ARM}_survival_cohort_{ARM}.csv"
    if cohort_csv.exists():
        out["stage1_cohort"] = pd.read_csv(cohort_csv, usecols=["DFCI_MRN"])["DFCI_MRN"].nunique()

    att2 = _load_json(root / "cohort_attrition.json")
    if att2:
        pre = att2.get("pre_consolidation_filters") or {}
        for k in ("n_before_early_prefilters", "n_after_parpi_prefilter",
                  "n_before_psa_prefilter", "n_after_psa_prefilter"):
            if k in pre:
                out[f"stage2_{k}"] = pre[k]
        for k in ("n_raw_longitudinal_patients", "n_with_labs",
                  "n_with_c61_diagnosis_date", "n_with_outcome_data",
                  "n_before_broad_icd_filter", "n_after_broad_icd_filter",
                  "n_with_highlighted_treatment_anchor", "n_output_patients"):
            if k in att2:
                out[f"stage2_{k}"] = att2[k]

    att3 = _load_json(root / "survival_analysis" / f"prediction_inputs_{ARM}" / "landmark_attrition.json")
    if att3:
        out["stage3_n_loaded_cohort"] = att3.get("n_loaded_cohort")
        for lm, n in (att3.get("eligible_by_landmark") or {}).items():
            out[f"stage3_lm{lm}_eligible"] = n

    return out

base_counts = _stage_counts(BASELINE_ROOT)
prof_counts = _stage_counts(PROFILE_ROOT)

order = list(base_counts) + [k for k in prof_counts if k not in base_counts]
funnel_rows = []
for k in order:
    a, b = base_counts.get(k), prof_counts.get(k)
    funnel_rows.append({
        "step": k,
        "ALL_2025_03": a,
        "PROFILE_DATA": b,
        "delta": (b - a) if (a is not None and b is not None) else None,
        "pct_change": round(100.0 * (b - a) / a, 1) if (a not in (None, 0) and b is not None) else None,
    })
funnel_df = pd.DataFrame(funnel_rows)
print("=== ALIGNED FUNNEL: stage 1 -> stage 2 -> stage 3 ===")
display(funnel_df)

In [ ]:
# VERDICT: the step with the largest negative delta is where to attack.
_f = funnel_df.dropna(subset=["delta"])
if _f.empty:
    print("VERDICT (S1): no comparable steps -- check artifact presence in section 0.")
else:
    worst = _f.loc[_f["delta"].idxmin()]
    print("=" * 78)
    print(f"VERDICT (S1): largest drop at '{worst['step']}': "
          f"{worst['ALL_2025_03']:,.0f} -> {worst['PROFILE_DATA']:,.0f} "
          f"({worst['delta']:+,.0f}, {worst['pct_change']:+.1f}%)")
    print("=" * 78)
    print("\nAll steps where profile_data < baseline:")
    display(_f.loc[_f["delta"] < 0].sort_values("delta"))

    # Where does the drop FIRST appear? That localizes the cause better than the max.
    first_neg = _f.loc[_f["delta"] < 0]
    if len(first_neg):
        fn = first_neg.iloc[0]
        print(f"\nFIRST step showing a loss: '{fn['step']}' ({fn['delta']:+,.0f}). "
              f"The cause lies at or before this step.")

## Section 2 -- Raw source comparison, before any pipeline logic

**Critical assertion:** `baseline_mrns - profile_mrns` should be **empty** for every
table, because `profile_data` merges in `ALL_2025_03`. A non-empty difference means
the loss happens upstream in `PROFILE_data_processing/compile_OncDRS_data.ipynb`,
not in this repo at all -- and the rest of this notebook is moot.

All reads go through `scan_source`, so CSV and parquet are compared on equal
(all-Utf8) footing.

In [ ]:
from data_preprocessing_common.oncdrs_sources import scan_source, resolve, TABLE_FILES

def _mrn_set_and_rows(path):
    """(n_rows, set of Int64 DFCI_MRN) for a raw source, via scan_source."""
    lf = scan_source(path)
    df = lf.select(
        pl.len().alias("n_rows"),
    ).collect()
    n_rows = int(df["n_rows"][0])
    mrns = (
        scan_source(path)
        .select(
            pl.col("DFCI_MRN").cast(pl.Float64, strict=False).cast(pl.Int64, strict=False)
        )
        .collect()["DFCI_MRN"]
        .drop_nulls()
        .unique()
        .to_list()
    )
    return n_rows, set(mrns)

RAW_MRNS = {}   # (variant, table) -> set
rows = []
for table in cp.SOURCE_TABLES:
    rec = {"table": table}
    for variant in (BASELINE_VARIANT, DATA_VARIANT):
        path = cp.DATA_VARIANTS[variant]["sources"][table]
        if not Path(path).exists():
            rec[f"{variant}_rows"] = None
            rec[f"{variant}_mrns"] = None
            continue
        try:
            n_rows, mrns = _mrn_set_and_rows(path)
        except Exception as e:
            print(f"  {variant}/{table}: read failed -- {type(e).__name__}: {e}")
            rec[f"{variant}_rows"] = None
            rec[f"{variant}_mrns"] = None
            continue
        RAW_MRNS[(variant, table)] = mrns
        rec[f"{variant}_rows"] = n_rows
        rec[f"{variant}_mrns"] = len(mrns)
    rows.append(rec)

raw_df = pd.DataFrame(rows)
print("=== RAW SOURCE SIZE ===")
display(raw_df)

In [ ]:
# The critical assertion: baseline MRNs must all be present in profile_data.
rows = []
for table in cp.SOURCE_TABLES:
    b = RAW_MRNS.get((BASELINE_VARIANT, table))
    p = RAW_MRNS.get((DATA_VARIANT, table))
    if b is None or p is None:
        continue
    only_b, only_p, both = b - p, p - b, b & p
    rows.append({
        "table": table,
        "baseline_only": len(only_b),
        "profile_only": len(only_p),
        "intersection": len(both),
        "SUPERSET_OK": len(only_b) == 0,
        "sample_baseline_only": sorted(only_b)[:5] if only_b else [],
    })

setdiff_df = pd.DataFrame(rows)
print("=== MRN SET RELATIONSHIP (baseline vs profile_data) ===")
display(setdiff_df)

violations = setdiff_df.loc[~setdiff_df["SUPERSET_OK"]] if len(setdiff_df) else pd.DataFrame()
print("=" * 78)
if len(violations):
    print("VERDICT (S2): SUPERSET VIOLATED -- profile_data is MISSING baseline MRNs.")
    print("The loss originates UPSTREAM, in PROFILE_data_processing/compile_OncDRS_data.ipynb.")
    print("Fix the merge there; sections 3-5 diagnose this repo and will not explain it.")
    display(violations[["table", "baseline_only", "sample_baseline_only"]])
else:
    print("VERDICT (S2): superset holds -- profile_data contains every baseline MRN")
    print("at the raw level. The loss is introduced by PIPELINE LOGIC in this repo.")
    print("Proceed to section 3 (date parsing).")
print("=" * 78)

In [ ]:
# Suspect #1: EHR_DIAGNOSIS.csv (singular) vs EHR_DIAGNOSES.parquet (plural).
# oncdrs_sources.TABLE_FILES maps the parquet as plural; the script argparse
# default is singular. A resolve()-based call and a default-flag call can reach
# different files.
print("=== ICD FILENAME RESOLUTION ===")
print(f"TABLE_FILES['EHR_DIAGNOSES'] = {TABLE_FILES['EHR_DIAGNOSES']}")
resolved = resolve("EHR_DIAGNOSES", cp._PROFILE_DATA_ROOT)
print(f"resolve('EHR_DIAGNOSES', PROFILE_DATA_ROOT) -> {resolved}  exists={resolved.exists()}")

print("\nCandidate ICD files actually present under the profile_data root:")
for cand in sorted(Path(cp._PROFILE_DATA_ROOT).glob("EHR_DIAGNOS*")):
    st = cand.stat()
    print(f"  {cand.name:36s} {st.st_size/1e6:9.1f} MB  {_dt.datetime.fromtimestamp(st.st_mtime)}")

print(f"\nPipeline is configured to read: "
      f"{cp.DATA_VARIANTS[DATA_VARIANT]['sources']['EHR_DIAGNOSES']}")
if not Path(cp.DATA_VARIANTS[DATA_VARIANT]["sources"]["EHR_DIAGNOSES"]).exists():
    print("!! That path does NOT exist -- resolve() fell through to the CSV basename.")

## Section 3 -- Date-parse audit (prime suspect)

For every `(table, date_column)` the pipeline parses, in **both** variants:

1. `n_non_null_raw` -- rows with a non-blank string
2. `n_parsed` -- rows surviving `str.to_datetime(strict=False)`, replicating the
   pipeline call exactly (`ComputeError` is itself a finding)
3. `parse_loss` and **`n_mrns_lost`** -- MRNs with >=1 non-null raw date but zero
   parsed dates. This is the number that matters.
4. **Format census** -- raw strings grouped by shape signature (digits->`9`,
   letters->`A`). **>1 shape in a profile_data column is the smoking gun**;
   baseline should show exactly one.

In [ ]:
# The exact (table, column) pairs the pipeline parses with str.to_datetime.
#   MED_START_DT      compile_COMPASS_cohort_data.py:522   -> row dropped at :524
#   START_DT          compile_COMPASS_cohort_data.py:267   -> silently fails to exclude
#   BIRTH_DT          compile_COMPASS_cohort_data.py:566   -> null AGE -> dropped in stage 3
#   HYBRID_DEATH_DT   compile_COMPASS_cohort_data.py:567
#   DERIVED_LAST_ALIVE_DATE  compile_COMPASS_cohort_data.py:568
#   SPECIMEN_COLLECT_DT      longitudinal lab dates
DATE_COLUMNS = [
    ("MEDICATIONS",                 "MED_START_DT"),
    ("EHR_DIAGNOSES",               "START_DT"),
    ("PT_INFO_STATUS_REGISTRATION", "BIRTH_DT"),
    ("PT_INFO_STATUS_REGISTRATION", "HYBRID_DEATH_DT"),
    ("PT_INFO_STATUS_REGISTRATION", "DERIVED_LAST_ALIVE_DATE"),
    ("LABS",                        "SPECIMEN_COLLECT_DT"),
]

BLANKS = ["", "NAN", "NONE", "NULL"]

def _shape_expr(col):
    """Normalized shape signature: digits -> 9, letters -> A, rest literal."""
    return (
        pl.col(col).cast(pl.Utf8).str.strip_chars()
        .str.replace_all(r"\d", "9")
        .str.replace_all(r"[A-Za-z]", "A")
    )

def audit_date_column(path, col):
    """Replicate the pipeline's parse and measure exactly what it loses."""
    lf = scan_source(path)
    if col not in lf.collect_schema().names():
        return {"status": "column_absent"}

    base = lf.select(
        pl.col("DFCI_MRN").cast(pl.Float64, strict=False).cast(pl.Int64, strict=False).alias("MRN"),
        pl.col(col).cast(pl.Utf8).str.strip_chars().alias("raw"),
    ).with_columns(
        (pl.col("raw").is_not_null() & ~pl.col("raw").str.to_uppercase().is_in(BLANKS)).alias("has_raw")
    )

    try:
        df = base.with_columns(
            pl.when(pl.col("has_raw"))
              .then(pl.col("raw").str.to_datetime(strict=False))
              .otherwise(None)
              .alias("parsed")
        ).collect()
        status = "ok"
    except Exception as e:
        # No inferable format at all -- every value is lost.
        n_raw = int(base.select(pl.col("has_raw").sum()).collect()[0, 0])
        return {"status": f"PARSE_RAISED: {type(e).__name__}",
                "n_non_null_raw": n_raw, "n_parsed": 0, "parse_loss": n_raw}

    n_raw = int(df["has_raw"].sum())
    n_parsed = int(df["parsed"].is_not_null().sum())

    # MRNs that had a date but ended up with none parsed -- the real casualties.
    per_mrn = (
        df.filter(pl.col("has_raw"))
          .group_by("MRN")
          .agg(pl.col("parsed").is_not_null().sum().alias("n_ok"))
    )
    n_mrns_with_raw = per_mrn.height
    n_mrns_lost = int((per_mrn["n_ok"] == 0).sum())

    # Format census over the values that actually carry a date.
    census = (
        df.filter(pl.col("has_raw"))
          .select(_shape_expr("raw").alias("shape"),
                  pl.col("parsed").is_not_null().alias("ok"))
          .group_by("shape")
          .agg(pl.len().alias("n"), pl.col("ok").sum().alias("n_parsed"))
          .sort("n", descending=True)
    )

    return {
        "status": status,
        "n_non_null_raw": n_raw,
        "n_parsed": n_parsed,
        "parse_loss": n_raw - n_parsed,
        "n_mrns_with_raw": n_mrns_with_raw,
        "n_mrns_lost": n_mrns_lost,
        "n_shapes": census.height,
        "census": census,
    }

In [ ]:
date_rows = []
CENSUS = {}
for table, col in DATE_COLUMNS:
    for variant in (BASELINE_VARIANT, DATA_VARIANT):
        path = cp.DATA_VARIANTS[variant]["sources"][table]
        if not Path(path).exists():
            continue
        res = audit_date_column(path, col)
        census = res.pop("census", None)
        if census is not None:
            CENSUS[(variant, table, col)] = census
        date_rows.append({"variant": variant, "table": table, "column": col, **res})
        print(f"  audited {variant:12s} {table:28s} {col}")

date_df = pd.DataFrame(date_rows)
print("\n=== DATE PARSE AUDIT ===")
display(date_df)

In [ ]:
print("=== FORMAT CENSUS (shape signature: digits->9, letters->A) ===")
print("Multiple shapes are only a problem if some parse to ZERO rows -- ISO variants like")
print("'9999-99-99' and '9999-99-99 99:99:99' are distinct shapes but both parse fine.")
print("Watch the n_parsed column, not the shape count.\n")
for (variant, table, col), census in CENSUS.items():
    n_shapes = census.height
    flag = "   <-- MULTIPLE FORMATS" if (n_shapes > 1 and variant == DATA_VARIANT) else ""
    print(f"--- {variant} / {table}.{col}: {n_shapes} shape(s){flag}")
    with pl.Config(tbl_rows=12):
        print(census.head(12))
    # Which shapes are being nulled entirely?
    dead = census.filter(pl.col("n_parsed") == 0)
    if dead.height:
        print(f"    !! shapes parsed to ZERO rows (silently discarded): "
              f"{dead['shape'].to_list()}  ({int(dead['n'].sum()):,} rows)")
    print()

In [ ]:
# The D/M-swap hazard: '03/04/2024' parsed under an inferred format silently
# becomes Apr 3 rather than failing. A distribution with no day > 12 is the tell.
print("=== DAY-OF-MONTH DISTRIBUTION (D/M swap check) ===")
for variant in (BASELINE_VARIANT, DATA_VARIANT):
    path = cp.DATA_VARIANTS[variant]["sources"]["MEDICATIONS"]
    if not Path(path).exists():
        continue
    try:
        d = (
            scan_source(path)
            .select(pl.col("MED_START_DT").cast(pl.Utf8).str.to_datetime(strict=False).alias("d"))
            .drop_nulls()
            .select(pl.col("d").dt.day().alias("day"))
            .collect()
        )
    except Exception as e:
        print(f"  {variant}: parse raised {type(e).__name__}")
        continue
    if d.height == 0:
        print(f"  {variant}: no parsed dates")
        continue
    max_day = int(d["day"].max())
    frac_gt12 = float((d["day"] > 12).mean())
    print(f"  {variant:12s} n={d.height:>10,}  max_day={max_day:2d}  frac(day>12)={frac_gt12:.3f}"
          + ("   <-- SUSPICIOUS: looks month-like, D/M may be swapped" if max_day <= 12 else ""))

    yr = (
        scan_source(path)
        .select(pl.col("MED_START_DT").cast(pl.Utf8).str.to_datetime(strict=False).dt.year().alias("y"))
        .drop_nulls().collect()["y"]
    )
    print(f"               year range {int(yr.min())}-{int(yr.max())}")

In [ ]:
# VERDICT for section 3.
print("=" * 78)
if date_df.empty:
    print("VERDICT (S3): no date columns audited -- check source availability.")
else:
    prof = date_df.loc[date_df["variant"] == DATA_VARIANT]
    base = date_df.loc[date_df["variant"] == BASELINE_VARIANT]

    prof_bad = prof.loc[prof.get("parse_loss", pd.Series(dtype=float)).fillna(0) > 0]
    base_bad = base.loc[base.get("parse_loss", pd.Series(dtype=float)).fillna(0) > 0]

    print(f"VERDICT (S3): profile_data columns losing rows to date parsing: {len(prof_bad)}")
    if len(prof_bad):
        display(prof_bad[["table", "column", "n_non_null_raw", "n_parsed",
                          "parse_loss", "n_mrns_lost", "n_shapes", "status"]])
        total_mrns_lost = int(prof_bad["n_mrns_lost"].fillna(0).max())
        print(f"\nWorst single column loses {total_mrns_lost:,} MRNs entirely.")
        print("Compare that to the section-1 shortfall. If comparable, DATE PARSING IS THE CAUSE.")
        print("\nFix: replace format-inferring str.to_datetime with a per-row parser")
        print("     (pd.to_datetime(errors='coerce'), or pl.coalesce over explicit formats)")
        print("     at all ~12 call sites, starting with compile_COMPASS_cohort_data.py:522.")
    else:
        print("No parse loss in profile_data -- date parsing is NOT the cause. Continue to section 4.")

    if len(base_bad):
        print(f"\nNOTE: baseline also loses rows ({len(base_bad)} columns) -- the audit's own")
        print("sanity check (verification step 3) expects zero here. Investigate the audit if so.")
print("=" * 78)

## Section 4 -- Stage 1 funnel reconstruction, per predicate

Stage 1 writes no attrition JSON (its funnel is `print`-only), so reconstruct its
MRN sets read-only for both variants, mirroring `main()` at
`compile_COMPASS_cohort_data.py:788-830`. The real functions are imported rather
than reimplemented, so this reconstruction cannot drift from the pipeline.

Ordering matters: `main()` filters medications to `all_cohort_mrns` via
`compile_cohort_tables` *before* `load_medications_for_survival` runs.

In [ ]:
sys.path.insert(0, str(cp.DATA_PREPROCESSING_DIR))
import compile_COMPASS_cohort_data as c1

def reconstruct_stage1(variant, verbose=True):
    """Mirror main() steps 1-3 read-only. No files are written."""
    src = cp.DATA_VARIANTS[variant]["sources"]
    out = {"variant": variant}

    icds = c1.load_and_explode_icd(src["EHR_DIAGNOSES"])
    out["n_icd_rows"] = icds.height

    non_prostate = c1.compute_non_prostate_primary_mrns(icds)
    prostate_excl, icd_excluded = c1.compute_prostate_cohort(icds, non_prostate)
    all_cohort_mrns = prostate_excl | icd_excluded
    out["all_cohort_mrns"] = all_cohort_mrns
    out["n_c61_cohort"] = len(all_cohort_mrns)

    # main() scopes medications to the C61 cohort before the survival load.
    meds = c1.filter_cohort(src["MEDICATIONS"], set(int(m) for m in all_cohort_mrns))
    out["n_med_rows_cohort_scoped"] = meds.height

    # Split load_medications_for_survival so the date filter is measured separately.
    keep = {m.upper() for m in c1.ARPI_ANCHOR_MEDS | c1.ADT_ANCHOR_MEDS | c1.PLATINUM_MEDS}
    named = meds.with_columns(
        pl.col("NCI_PREFERRED_MED_NM").cast(pl.Utf8).str.to_uppercase().str.strip_chars()
          .alias("NCI_PREFERRED_MED_NM")
    ).filter(pl.col("NCI_PREFERRED_MED_NM").is_in(list(keep)))
    out["n_anchor_med_rows_before_date_filter"] = named.height
    out["n_mrns_before_date_filter"] = named["DFCI_MRN"].cast(pl.Float64, strict=False)\
        .cast(pl.Int64, strict=False).drop_nulls().n_unique()

    dated = named.with_columns(
        pl.col("MED_START_DT").str.to_datetime(strict=False).alias("MED_START_DT")
    ).filter(pl.col("MED_START_DT").is_not_null())
    out["n_anchor_med_rows_after_date_filter"] = dated.height
    out["n_mrns_after_date_filter"] = dated["DFCI_MRN"].cast(pl.Float64, strict=False)\
        .cast(pl.Int64, strict=False).drop_nulls().n_unique()
    out["med_rows_lost_to_date_parse"] = named.height - dated.height

    meds_for_survival = c1.load_medications_for_survival(meds)
    adt_anchor_df = c1.compute_treatment_anchor(meds_for_survival, meds_set=c1.ADT_ANCHOR_MEDS)
    adt_entry_mrns = set(adt_anchor_df[c1.ID_COL].drop_nulls().to_list())
    out["adt_entry_mrns"] = adt_entry_mrns
    out["n_adt_entry"] = len(adt_entry_mrns)

    post_adt = c1.compute_post_adt_exclusion_cancer_mrns(icds, adt_anchor_df)
    out["post_adt_exclusion_mrns"] = post_adt
    out["n_post_adt_exclusion_in_cohort"] = len(all_cohort_mrns & post_adt)

    eligible = (all_cohort_mrns & adt_entry_mrns) - post_adt
    out["eligible_mrns"] = eligible
    out["n_c61_and_adt"] = len(all_cohort_mrns & adt_entry_mrns)
    out["n_eligible"] = len(eligible)

    if verbose:
        print(f"  {variant}: C61={out['n_c61_cohort']:,} "
              f"ADT-entry={out['n_adt_entry']:,} "
              f"C61&ADT={out['n_c61_and_adt']:,} "
              f"eligible={out['n_eligible']:,}")
    return out

print("Reconstructing Stage 1 (this reads the full raw sources; may take a few minutes)...\n")
S1 = {}
for variant in (BASELINE_VARIANT, DATA_VARIANT):
    print(f"--- {variant} ---")
    S1[variant] = reconstruct_stage1(variant)

In [ ]:
STEP_KEYS = [
    "n_icd_rows",
    "n_c61_cohort",
    "n_med_rows_cohort_scoped",
    "n_anchor_med_rows_before_date_filter",
    "n_mrns_before_date_filter",
    "n_anchor_med_rows_after_date_filter",
    "n_mrns_after_date_filter",
    "med_rows_lost_to_date_parse",
    "n_adt_entry",
    "n_c61_and_adt",
    "n_post_adt_exclusion_in_cohort",
    "n_eligible",
]

rows = []
for k in STEP_KEYS:
    a, b = S1[BASELINE_VARIANT].get(k), S1[DATA_VARIANT].get(k)
    rows.append({
        "step": k, "ALL_2025_03": a, "PROFILE_DATA": b,
        "delta": (b - a) if (a is not None and b is not None) else None,
        "pct_change": round(100.0 * (b - a) / a, 1) if (a not in (None, 0) and b is not None) else None,
    })
stage1_funnel = pd.DataFrame(rows)
print("=== STAGE 1 FUNNEL, PER PREDICATE ===")
display(stage1_funnel)

# Validation (verification step 4): the reconstruction must reproduce the
# baseline run's actual cohort CSV row count.
base_csv = Path(BASELINE_ROOT) / f"prostate_{ARM}_survival_cohort_{ARM}.csv"
if base_csv.exists():
    actual = pd.read_csv(base_csv, usecols=["DFCI_MRN"])["DFCI_MRN"].nunique()
    recon = S1[BASELINE_VARIANT]["n_eligible"]
    ok = actual == recon
    print(f"\nRECONSTRUCTION CHECK (baseline): actual cohort CSV={actual:,} "
          f"vs reconstructed eligible={recon:,}  {'MATCH' if ok else 'MISMATCH'}")
    if not ok:
        print("  !! Reconstruction does not mirror main(). Treat section 4/5 numbers with caution.")

In [ ]:
# Separate the two competing explanations at the ADT-entry and exclusion steps.
b, p = S1[BASELINE_VARIANT], S1[DATA_VARIANT]

lost_at_adt = (b["all_cohort_mrns"] & b["adt_entry_mrns"]) - (p["all_cohort_mrns"] & p["adt_entry_mrns"])
gained_at_adt = (p["all_cohort_mrns"] & p["adt_entry_mrns"]) - (b["all_cohort_mrns"] & b["adt_entry_mrns"])
extra_excluded = (p["post_adt_exclusion_mrns"] & p["all_cohort_mrns"]) - b["post_adt_exclusion_mrns"]

print("=== WHERE THE SETS DIVERGE ===")
print(f"  in baseline C61&ADT but NOT in profile C61&ADT : {len(lost_at_adt):,}   <-- BUG-shaped")
print(f"  in profile C61&ADT but NOT in baseline         : {len(gained_at_adt):,}   (expected: more releases)")
print(f"  newly post-ADT-excluded under profile_data     : {len(extra_excluded):,}   <-- LEGITIMATE")
print()
print("Interpretation:")
print("  'lost_at_adt' patients had an ADT record in a SUBSET of the data but lost it in the")
print("  SUPERSET. That is impossible without a bug -- almost certainly the date-parse filter.")
print("  'extra_excluded' is legitimate: more releases -> more diagnosis rows -> more patients")
print("  correctly trip the post-ADT exclusion cancers. Do not 'fix' that.")

LOST_AT_ADT = lost_at_adt
FINAL_LOST = b["eligible_mrns"] - p["eligible_mrns"]
print(f"\n  final eligible lost (baseline - profile): {len(FINAL_LOST):,}")
print(f"    of which explained by extra exclusions : {len(FINAL_LOST & extra_excluded):,}")
print(f"    of which lost at the ADT-entry gate    : {len(FINAL_LOST & lost_at_adt):,}")
print(f"    unaccounted                            : "
      f"{len(FINAL_LOST - extra_excluded - lost_at_adt):,}")

In [ ]:
# Suspect #2: medication-name spelling drift between releases.
def anchor_name_counts(variant):
    path = cp.DATA_VARIANTS[variant]["sources"]["MEDICATIONS"]
    return (
        scan_source(path)
        .select(pl.col("NCI_PREFERRED_MED_NM").cast(pl.Utf8).str.to_uppercase().str.strip_chars().alias("nm"))
        .group_by("nm").agg(pl.len().alias("n"))
        .collect()
    )

INTEREST = c1.ARPI_ANCHOR_MEDS | c1.ADT_ANCHOR_MEDS | c1.PLATINUM_MEDS
name_counts = {v: anchor_name_counts(v) for v in (BASELINE_VARIANT, DATA_VARIANT)}

bn = {r["nm"]: r["n"] for r in name_counts[BASELINE_VARIANT].iter_rows(named=True)}
pn = {r["nm"]: r["n"] for r in name_counts[DATA_VARIANT].iter_rows(named=True)}

rows = []
for nm in sorted(INTEREST):
    rows.append({"med_name": nm, "in_baseline": bn.get(nm, 0), "in_profile": pn.get(nm, 0),
                 "MATCHED_BOTH": nm in bn and nm in pn})
med_match_df = pd.DataFrame(rows)
print("=== ANCHOR/PLATINUM MED NAME MATCHING ===")
display(med_match_df)

missing_in_profile = med_match_df.loc[(med_match_df["in_baseline"] > 0) & (med_match_df["in_profile"] == 0)]
if len(missing_in_profile):
    print("!! Names present in baseline but ABSENT in profile_data -- spelling drift:")
    display(missing_in_profile)

# Near-miss spellings only present in one source (substring match on the drug stem).
stems = sorted({nm.split()[0] for nm in INTEREST if len(nm.split()[0]) > 5})
near = []
for nm in set(pn) | set(bn):
    if nm in INTEREST or not nm:
        continue
    if any(s in nm for s in stems):
        near.append({"med_name": nm, "in_baseline": bn.get(nm, 0), "in_profile": pn.get(nm, 0)})
if near:
    print("\nUnmatched names containing an anchor-drug stem (candidate missed spellings):")
    display(pd.DataFrame(near).sort_values("in_profile", ascending=False).head(30))

In [ ]:
print("=" * 78)
_s1 = stage1_funnel.set_index("step")["delta"].to_dict()
if _s1.get("n_eligible") is not None and _s1["n_eligible"] < 0:
    print(f"VERDICT (S4): Stage 1 loses {abs(_s1['n_eligible']):,} eligible patients under profile_data.")
    print(f"  med rows lost to date parsing: baseline="
          f"{S1[BASELINE_VARIANT]['med_rows_lost_to_date_parse']:,} "
          f"profile={S1[DATA_VARIANT]['med_rows_lost_to_date_parse']:,}")
    print(f"  MRNs losing ADT entry despite a superset of data: {len(LOST_AT_ADT):,}  <-- bug-shaped")
    print(f"  legitimately newly excluded (post-ADT cancers)  : {len(extra_excluded):,}")
else:
    print("VERDICT (S4): Stage 1 does NOT lose patients. The drop is downstream (stage 2/3);")
    print("  section 5 will fall through to the landmark-level attribution.")
print("=" * 78)

## Section 5 -- Per-MRN attribution: where exactly did each lost patient die?

The payoff. Take the patients present in the baseline cohort but absent from
`profile_data`, and classify each into exactly one bucket by interrogating the
`profile_data` raw sources directly.

In [ ]:
# Lost set, preferring the actual written cohort CSVs over the reconstruction.
def _cohort_mrns(root):
    p = Path(root) / f"prostate_{ARM}_survival_cohort_{ARM}.csv"
    if not p.exists():
        return None
    return set(pd.read_csv(p, usecols=["DFCI_MRN"])["DFCI_MRN"].dropna().astype(int))

base_cohort = _cohort_mrns(BASELINE_ROOT)
prof_cohort = _cohort_mrns(PROFILE_ROOT)

if base_cohort is not None and prof_cohort is not None:
    LOST = base_cohort - prof_cohort
    print(f"From written cohort CSVs: baseline={len(base_cohort):,} profile={len(prof_cohort):,} "
          f"lost={len(LOST):,}")
else:
    LOST = FINAL_LOST
    print(f"Cohort CSV missing in a root; falling back to reconstruction. lost={len(LOST):,}")

SAMPLE_N = 20
rng = np.random.default_rng(0)
sample = sorted(rng.choice(sorted(LOST), size=min(SAMPLE_N, len(LOST)), replace=False).tolist()) if LOST else []
print(f"Sampling {len(sample)} lost MRNs for attribution.")

In [ ]:
def attribute_lost_mrns(mrns):
    """Classify each lost MRN into exactly one failure bucket, with evidence."""
    if not mrns:
        return pd.DataFrame()

    src = cp.DATA_VARIANTS[DATA_VARIANT]["sources"]
    mrn_list = [int(m) for m in mrns]

    def _scoped(table, cols):
        lf = scan_source(src[table])
        names = lf.collect_schema().names()
        keep = [c for c in cols if c in names]
        return (
            lf.with_columns(
                pl.col("DFCI_MRN").cast(pl.Float64, strict=False).cast(pl.Int64, strict=False).alias("MRN")
            )
            .filter(pl.col("MRN").is_in(mrn_list))
            .select(["MRN"] + keep)
            .collect()
        )

    meds = _scoped("MEDICATIONS", ["NCI_PREFERRED_MED_NM", "MED_START_DT"])
    meds = meds.with_columns(
        pl.col("NCI_PREFERRED_MED_NM").cast(pl.Utf8).str.to_uppercase().str.strip_chars().alias("nm")
    )
    adt_meds = meds.filter(pl.col("nm").is_in(list(c1.ADT_ANCHOR_MEDS)))
    adt_meds = adt_meds.with_columns(
        pl.col("MED_START_DT").cast(pl.Utf8).str.to_datetime(strict=False).alias("parsed"),
        _shape_expr("MED_START_DT").alias("shape"),
    )

    icds = _scoped("EHR_DIAGNOSES", ["DIAGNOSIS_ICD10_CD", "START_DT"])
    labs = _scoped("LABS", ["TEST_TYPE_CD"])

    per_mrn_adt = {
        r["MRN"]: r for r in adt_meds.group_by("MRN").agg(
            pl.len().alias("n_adt_rows"),
            pl.col("parsed").is_not_null().sum().alias("n_adt_dated"),
            pl.col("shape").unique().alias("shapes"),
            pl.col("MED_START_DT").first().alias("example_raw"),
        ).iter_rows(named=True)
    }
    psa_counts = {
        r["MRN"]: r["n"] for r in labs.filter(
            pl.col("TEST_TYPE_CD").cast(pl.Utf8).is_in(list(c1.BROAD_PSA_CODES))
        ).group_by("MRN").agg(pl.len().alias("n")).iter_rows(named=True)
    }
    c61 = set(
        icds.filter(
            pl.col("DIAGNOSIS_ICD10_CD").cast(pl.Utf8).str.to_uppercase().str.strip_chars()
              .str.starts_with("C61")
        )["MRN"].unique().to_list()
    )
    post_adt = S1[DATA_VARIANT]["post_adt_exclusion_mrns"]

    rows = []
    for m in mrn_list:
        a = per_mrn_adt.get(m)
        n_psa = psa_counts.get(m, 0)
        if m not in c61:
            bucket, ev = "no_c61_diagnosis", "no C61 code in profile_data EHR_DIAGNOSES"
        elif a is None or a["n_adt_rows"] == 0:
            bucket, ev = "no_adt_record", "no ADT anchor-drug row in profile_data MEDICATIONS"
        elif a["n_adt_dated"] == 0:
            bucket = "date_parse_failure"
            ev = (f"{a['n_adt_rows']} ADT row(s), ALL dates unparseable; "
                  f"shapes={a['shapes']} example={a['example_raw']!r}")
        elif m in post_adt:
            bucket, ev = "post_adt_exclusion", "post-ADT exclusion cancer diagnosed after ADT start"
        elif n_psa < c1.MIN_PSA_COUNT:
            bucket, ev = "psa_count_below_5", f"broad PSA records={n_psa} < {c1.MIN_PSA_COUNT}"
        else:
            bucket = "unexplained"
            ev = (f"C61 yes, ADT rows={a['n_adt_rows']} dated={a['n_adt_dated']}, "
                  f"PSA={n_psa}, not post-ADT-excluded")
        rows.append({"DFCI_MRN": m, "failed_predicate": bucket, "evidence": ev})

    return pd.DataFrame(rows)

attribution = attribute_lost_mrns(sample)
if not attribution.empty:
    print("=== PER-MRN ATTRIBUTION (sample) ===")
    display(attribution)
    print("\n=== BUCKET HISTOGRAM ===")
    hist = attribution["failed_predicate"].value_counts()
    display(hist)
    explained = 1 - (hist.get("unexplained", 0) / len(attribution))
    print(f"\nExplained: {explained:.0%} of sampled lost MRNs "
          f"({'OK -- >=80% target met' if explained >= 0.8 else 'BELOW 80% -- a suspect is missing'})")
else:
    print("No lost MRNs at stage 1 -- falling through to landmark-level attribution below.")

In [ ]:
# Fallback: if nothing was lost at stage 1, pin the drop to a landmark condition
# using landmark_mrn_availability.csv, the only per-MRN artifact stage 3 writes.
if attribution.empty or len(LOST) == 0:
    def _avail(root):
        p = Path(root) / "survival_analysis" / f"prediction_inputs_{ARM}" / "landmark_mrn_availability.csv"
        return pd.read_csv(p) if p.exists() else None

    av_b, av_p = _avail(BASELINE_ROOT), _avail(PROFILE_ROOT)
    if av_b is None or av_p is None:
        print("landmark_mrn_availability.csv missing in a root -- cannot do landmark attribution.")
    else:
        print("=== LANDMARK-LEVEL ELIGIBILITY (per-MRN artifact) ===")
        elig_cols = [c for c in av_b.columns if c.startswith("eligible_landmark_")]
        rows = []
        for c in elig_cols + ["eligible_all_landmarks", "included_all_landmarks"]:
            if c in av_b.columns and c in av_p.columns:
                a, b_ = int(av_b[c].sum()), int(av_p[c].sum())
                rows.append({"condition": c, "ALL_2025_03": a, "PROFILE_DATA": b_, "delta": b_ - a})
        display(pd.DataFrame(rows))

        for c in elig_cols:
            sb = set(av_b.loc[av_b[c] == 1, "DFCI_MRN"])
            sp = set(av_p.loc[av_p[c] == 1, "DFCI_MRN"])
            print(f"  {c}: lost={len(sb - sp):,}  gained={len(sp - sb):,}")
        print("\nA landmark that loses patients while earlier stages did not points at the")
        print("nine ANDed conditions in survival_common/cohort.py:256-297 (notna + strictly > 0),")
        print("most often a null t_* driven by an unparseable upstream date.")
else:
    print("Stage 1 attribution succeeded; landmark fallback not needed.")

## Section 6 -- Summary

Ranked causes, separating **bugs** (fix these) from **legitimate differences**
(do not 'fix' these -- more releases genuinely means more patients correctly
tripping the post-ADT exclusion).

In [ ]:
summary_rows = []

# Bug: date-parse nulling.
if not date_df.empty:
    prof = date_df.loc[date_df["variant"] == DATA_VARIANT]
    worst = prof.loc[prof["parse_loss"].fillna(0) > 0].sort_values("n_mrns_lost", ascending=False)
    for _, r in worst.iterrows():
        summary_rows.append({
            "kind": "BUG",
            "cause": f"date parse nulls {r['table']}.{r['column']}",
            "patients_explained": int(r["n_mrns_lost"]) if pd.notna(r["n_mrns_lost"]) else None,
            "evidence": f"S3: {int(r['parse_loss']):,} rows lost, {int(r['n_shapes'])} format shape(s)",
            "fix": "per-row parser (pd.to_datetime errors='coerce' / pl.coalesce over formats)",
        })

# Bug: MRNs losing ADT entry despite a data superset.
try:
    if len(LOST_AT_ADT):
        summary_rows.append({
            "kind": "BUG",
            "cause": "MRNs lose ADT entry in the superset",
            "patients_explained": len(LOST_AT_ADT),
            "evidence": "S4: present in baseline C61&ADT, absent in profile C61&ADT",
            "fix": "same date-parse fix at compile_COMPASS_cohort_data.py:522-524",
        })
except NameError:
    pass

# Legitimate: extra exclusions from extra releases.
try:
    if len(extra_excluded):
        summary_rows.append({
            "kind": "LEGITIMATE",
            "cause": "more releases -> more post-ADT exclusion cancers",
            "patients_explained": len(extra_excluded),
            "evidence": "S4: newly post-ADT-excluded under profile_data",
            "fix": "none -- correct behavior, do not change",
        })
except NameError:
    pass

# Upstream: superset violation.
try:
    if len(violations):
        summary_rows.append({
            "kind": "UPSTREAM",
            "cause": "profile_data missing baseline MRNs at raw level",
            "patients_explained": int(violations["baseline_only"].max()),
            "evidence": "S2: superset assertion failed",
            "fix": "PROFILE_data_processing/compile_OncDRS_data.ipynb merge logic",
        })
except NameError:
    pass

summary_df = pd.DataFrame(summary_rows)
print("=== RANKED CAUSES ===")
if summary_df.empty:
    print("No causes identified -- review the per-section verdicts above.")
else:
    display(summary_df.sort_values(["kind", "patients_explained"], ascending=[True, False]))

out_path = DIAG_OUT / f"compass_diagnostics_summary_{ARM}.csv"
if not summary_df.empty:
    summary_df.to_csv(out_path, index=False)
    print(f"\nWrote {out_path}")

In [ ]:
# Verification step 2: confirm this notebook wrote nothing under either data root.
changed = []
for name, root in (("baseline", BASELINE_ROOT), ("profile", PROFILE_ROOT)):
    before = MTIME_SNAPSHOT[name]
    after = _mtime_snapshot(root)
    for path, mt in after.items():
        if path not in before:
            changed.append((name, path, "CREATED"))
        elif before[path] != mt:
            changed.append((name, path, "MODIFIED"))
    for path in before:
        if path not in after:
            changed.append((name, path, "DELETED"))

print("=== READ-ONLY CHECK ===")
if changed:
    print(f"!! {len(changed)} file(s) changed under the data roots -- NOT read-only:")
    for c in changed[:25]:
        print("   ", c)
else:
    print("PASS: no file under either data root was created, modified, or deleted.")